
## By Knowing this notebook, we can become an eligible "Data Egress Developer/Engineer"


### We are writing data in 
- Structured(csv), 
- Semi Structured(JSON/XML), 
- Serialized files (orc/parquet/delta)(Datalake)


###Table (delta/hive) (Lakehouse) format

### Let's get some data we have already...

In [0]:
print(spark)
name='franklin'


### this is going to get the spark session already object created by Databricks.

In [0]:
from pyspark.sql.session import SparkSession

spark1=SparkSession.builder.appName("Spark DataFrames").getOrCreate()
print(spark1)

In [0]:
#Extract

ingest_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header",
                            header=True,
                            sep=',',
                            inferSchema=True,
                            samplingRatio=0.10
                        )




### Schema migration/Data Conversion from CSV to Delta (ingested data into our DELTALAKE (DATALAKE))


### Extract (source file sits directly under dir1_dropme, not under data/)

In [0]:
from pyspark.sql.functions import col

ingest_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs_header",
                            header=True,
                            sep=',',
                            inferSchema=True,
                            samplingRatio=0.10
                        )

ingest_df1.write.format("delta") \
                .option("mergeSchema", "true") \
                .save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/ingestion_volume/target/deltaoutput/",
                        mode="overwrite"
                    )


### Writing the data in Builtin - different file formats & different targets (all targets in this world we can write the data also...)


### 1. Writing in csv (structured data (2D data Table/Frames with rows and columns)) format with few basic options listed below (Schema (structure) Migration)


custid,fname,lname,age,profession -> custid~fname~lname~prof~age
- header
- sep
- mode


### We are performing schema migration from comma to tilde delimiter.

In [0]:
display(ingest_df1.take(3))#Showing data in the edgenode

In [0]:
ingest_df1.write.csv(path="dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/csvout",
                     sep='~',
                     header=True,
                     mode='overwrite'
                    )#writing data to dbfs (DATALAKE)


### 4 modes of writing - append,overwrite,ignore,error


### We are performing schema migration/Data conversion by applying some transformations (this is our bread and butter that we learn exclusively further)

In [0]:
#Transform

transformed_df=ingest_df1.select("custid","fname","lname","profession","age") \
                         .withColumnRenamed("profession","prof")#DSL transformation (not for now...)

In [0]:
#Load

transformed_df.write.csv(path="/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/ingestion_volume/target/csvout_zip/",
                         sep='~',
                         header=True,
                         mode='overwrite',
                         compression='gzip'
                        )


## 2. Writing in json format with few basic options listed below.


- We did a schema migration and data conversion from csv to json format (ie structued to semi structured format)
- json - we learn a lot subsequently (nested/hierarchical/complex/multiline...), 
- what is json - fundamentally it is a dictionary of dictionaries
- json - java script object notation
- Standard json format (can't be changed) - {"k1":"string value","k2":numbervalue,"k3":v2} where key has to be unique & enclosed in double quotes and value can be anything
- **when to go with json or benifits** - 
- a. If we have data in a **semistructure** format (with variable data format with dynamic schema)
- eg. {"custid":4000001,"profession":"Pilot","age":55,"city":"NY"}<br>
      {"age":"55","custid":4000001,"fname":"Kristina","lname":"Chung","prof":"Pilot"}
- b. **Number of columns**, column names or the data types or the **order of columns** can be different
- c. json will be provided by the sources if the data is **dynamic** in nature **(not sure about number or order of columns)** or if the data is api response (HTTP WEBAPI Source) in nature.
- d. **json is a efficient data format** (serialized/encoded) for performing data exchange between applications via network & good for parsing also & good for object by object operations (row by row operation in realtime fashion eg. amazon click stream operations)
- e. json can be used to **group or create hierarchy of data in a complex or in a nested format** eg. https://randomuser.me/api/


### Schema Migration / Data Conversion from Structured to SemiStructured format..


- Structured -> SemiStruct...
- custid,fname,lname,age,profession -> {"custid":4000001,"fname":"Kristina","lname":"Chung","prof":"Pilot","age":55}

In [0]:
ingest_df1.write.json(path="dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/jsonout",
                        mode='append'
                    )


## 3.Serialization (encoding in a more optimized fashion) & Deserialization File formats (Binary/Brainy File formats)


**Data Mechanics:**

1. **encoding/decoding(machine format)** - converting the data from human readable format to machine understandable format for performant data transfer (eg. Network transfer of data will be encoded)
2. ***compression/uncompression(encoding+space+time)** - shrinking the data in some format using some libraries (tradeoff between time and size) (eg. Compress before store or transfer) - snappy is a good compression tech used in bigdata platform
3. **encryption (encoding+security)** - Addition to encoding, encryption add security hence data is (performant+secured) (using some algos - SHA/MD5/AES/DES/RSA/DSA..)
4. ***Serialization (applicable more for bigdata)** - Serialization is encoding + performant by saving space + processing intelligent bigdata format - Fast, Compact, Interoperable, Extensible (additional configs), Scalable (cluster compute operations), Secured (binary format)..
5. ***masking** - Encoding of data (in some other format not supposed to be machine format) which should not be allowed to decode (used for security purpose)


**What are the (builtin) serialized file formats we are going to learn?**

- orc
- parquet
- delta(databricks properatory)


- We did a schema migration and data conversion from csv/json to serialized data format (ie structued to sturctured(internall binary unstructured) format)
- We learn/use a lot/heavily subsequently
- what is serialized - fundamentally they are intelligent/encoded/serialized/binary data formats applied with lot of optimization & space reduction strategies.. (encoded/compressed/intelligent)
- orc - optimized row column format (Columnar formats)
- parquet - tiled data format (Columnar formats)
- delta(databricks properatory) enriched parquet format - Delta (modified/changes) operations can be performed (ACID property (DML))
- format - serialized/encoded , we can't see with mere eyes, only some library is used deserialized/decoded data can be accessed as structured data
- **when to go with serialized or benifits** - 
- a. For storage benifits for eg. orc will save 65+% of space for eg. if i store 1gb data it occupy 350mb space, with compression (snappy) it can improved more...
- b. For processing optimization. Orc/parquet/delta will provide the required data alone if you query using Pushdown optimization .
- c. Interoperability feature - this data format can be understandable in multiple environments for eg. bigquery can parse this data.
- d. Secured
- **In the projects/environments when to use what fileformats - we learn in detail later...


| Format  | Schema Type              | Storage Efficiency | Analytics/Transformation Performance | Updates Supported |
|--------|--------------------------|--------------------|-----------------------|------------------|
| CSV    | Structured               | Low                | Slow                  | No               |
| JSON   | Semi-structured           | Low                | Slow                  | No               |
| ORC    | Structured / Striped      | High               | Fast                  | Limited          |
| Parquet| Structured / Nested       | High               | Very Fast             | Limited          |
| Delta  | Structured / Evolving     | High               | Very Fast             | Highly           |
| XML    | Semi-structured           | Low                | Slow                  | No               |


### Since orc is an optimized Row & Column format data, it automatically preserves column names, datatype, delimiters & compression

In [0]:
ingest_df1.write.orc(path="dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/orcout",
                        mode='overwrite'
                    )

In [0]:
#by default orc/parquet uses snappy compression

df1=spark.read.orc("dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/orcout")

display(df1.take(2)) #uncompression + deserialization
df1.printSchema()


### Orc/Parquet follows WORM feature (Write Once Read Many) - No partial updates or partial deletes by using update or delete file/table.

### Since parquet is also an optimized Row & Column format data, it automatically preserves column names, datatype, delimiters & compression.

In [0]:
ingest_df1.write.mode("overwrite") \
                .option("compression","gzip") \
                .parquet(path="dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/parquetout")

In [0]:
df1=spark.read.parquet("dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/parquetout")

df1.show(2)#uncompression + deserialization
df1.printSchema()

In [0]:
ingest_df1.write.format("delta") \
                .save("dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/deltaout",
                                mode='overwrite'
                )

In [0]:
spark.read.format("parquet") \
          .load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/ingestion_volume/target/parquetoutput/") \
          .show(2)


### 4.Table Load Operations - Building LAKEHOUSE ON TOP OF DATALAKE


Can we do SQL operations directly on the tables like a database or datawarehouse? or Can we build a Lakehouse in Databricks?
- We learn/use a lot/heavily subsequently, 
- what is Lakehouse - A SQL/Datawarehouse/Query layer on top of the Datalake is called Lakehouse
- We have different lakehouses which we are going to learn further - 
1. delta tables (lakehouse) in databricks
2. hive in onprem
3. bigquery in GCP
4. synapse in azure
5. athena in aws
- **when to go with lakehouse** - 
- a. Transformation
- b. Analysis/Analytics
- c. AI/BI
- d. Literally we are going to learn SQL & Advanced SQL


### We are building delta tables in databricks (we are building hive tables in onprem/we are building bq tables in gcp...)

saveastable (named notation/named arguments)


### Table
cid,prof,age,fname,lname


### mapping

cid,prof,age,fname,lname

In [0]:

ingest_df1.write.saveAsTable("catalog1_dropme.schema1_dropme.lh_custtbl",
                             mode='overwrite'
                        )
#display(spark.sql("show create table catalog1_dropme.schema1_dropme.lh_custtbl"))


### 1. insertinto function can be used as like saveAstable with few differences



- it works only if the target table exist
- it works by creating insert statements in the behind(not bulk load), hence it is slow, hence we have use for small dataset (safely only if table exists)
- it will load the data from the dataframe by using position, not by using name..
- insertInto (positional notation/positional arguments)


### Table

cid,prof,age,fname,lname


### mapping

cid,fname,lname,age,prof

In [0]:
ingest_df1.write.insertInto("catalog1_dropme.schema1_dropme.lh_custtbl",
                                overwrite=True
                        )

In [0]:
ingest_df1.write.format("delta") \
                .save("dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/outputdata/deltaout2")


###I am using spark engine to pull the data from the lakehouse table backed by dbfs (s3) (datalake) where data in delta format(deltalake) 

In [0]:
display(spark.sql("select * from catalog1_dropme.schema1_dropme.lh_custtbl"))#sparkengine+lakehouse+datalake(deltalake)


### 5. XML Format - Semi structured data format (most of the json features can be applied in xml also, but in DE world not so famous like json)


- Used rarely on demand (by certain target/source systems eg. mainframes)
- Can be related with json, but not so much efficient like json
- Databricks provides xml as a inbuild function

In [0]:
ingest_df1.write.xml("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/ingestion_volume/target/xmloutput/",
                        mode="ignore",
                        rowTag="cust"
                    )


### Modes in Writing:


1. **Append** - Adds the new data to the existing data. It does not overwrite anything.
2. **Overwrite** - Replaces the existing data entirely at the destination.
3. **ErrorIfexist**(default) - Throws an error if data already exists at the destination.
4. **Ignore** - Skips the write operation if data already exists at the destination.


### What are all the overall functions/options we used in this notebook, for learning fundamental spark dataframe WRITE operations in different formats and targets?


1. We learned dozen of functions (out of 18 functions) in the write module with minimum options...
2. Functions we learned are (Datalake functions - csv/json/xml/orc/parquet+delta), (Lakehouse functions - saveAsTable/insertInto), (additional options - format/save/option/options/mode).
3. We have few more performance optimization/advanced options available (jdbc (we learn this soon in the name of foreign catalog), partitionBy,ClusterBy,BucketBy,SortBy,text)
4. Few of the important read options under csv such as header, sep, mode(append/overwrite/error/ignore), toDF.
5. Few additional options such as compression, different file formats...